# Indexing for RAG

Before an LLM can answer questions about your documents, those documents have to be turned into something searchable. That's **indexing**, and it has four steps:

1. **Load** the raw documents (PDFs, HTML, text…)
2. **Split** them into bite-sized chunks
3. **Embed** each chunk into a numeric vector
4. **Store** the vectors in a vector store you can query later

When a user asks a question, you embed the question too and find the chunks closest to it.

> Source: [LangChain Indexing concepts](https://docs.langchain.com/oss/python/concepts/rag)

## Setup

Load environment variables from your `.env` file (you should have `AZURE_OPENAI_API_KEY` set there). The Azure endpoint is filled in directly on the embeddings model further down.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

print("Environment loaded.")

## The `Document` abstraction

LangChain represents every chunk of text as a `Document` — a `page_content` string plus a `metadata` dict. You'll see this object everywhere in RAG.

In [ ]:
from langchain_core.documents import Document

example = Document(
    page_content="Wind turbines convert kinetic energy from wind into electricity.",
    metadata={"source": "intro-notes"},
)
example

## Step 1: Load a PDF

`PyPDFLoader` produces one `Document` per page. We'll use the Vestas Annual Report 2024 throughout these exercises.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "https://www.vestas.com/content/dam/vestas-com/global/en/investor/reports-and-presentations/financial/2024/fy-2024/Vestas%20Annual%20Report%202024.pdf.coredownload.inline.pdf"
)
docs = loader.load()

print(f"Loaded {len(docs)} pages.")
print(f"\nFirst page preview:\n{docs[0].page_content[:200]}")

## Step 2: Split into chunks

Whole pages are usually too big to feed to the LLM. `RecursiveCharacterTextSplitter` cuts them into roughly equal-sized chunks — with some `chunk_overlap` so a sentence isn't ripped from its context.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
all_splits = splitter.split_documents(docs)

print(f"Split {len(docs)} pages into {len(all_splits)} chunks.")

## Step 3: Create the embeddings model

Embeddings turn each chunk into a vector. We use Azure-hosted `text-embedding-3-large`. Paste your Azure endpoint into `azure_endpoint=...` below.

In [ ]:
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    azure_endpoint="",  # TODO: paste your Azure endpoint here
)

# Quick sanity check: embed a short string and look at the dimensionality.
vector = embeddings.embed_query("hello world")
print(f"Vector length: {len(vector)}")

## Step 4: Store the chunks in a vector store

For exercises we use `InMemoryVectorStore` — it lives in RAM and disappears when the kernel restarts. In production you'd swap this for Pinecone, Chroma, pgvector, etc.

`add_documents` embeds every chunk and stores it. This is the slow step — it makes one API call per chunk.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(all_splits)

print(f"Indexed {len(ids)} chunks.")

## Step 5: Search!

`similarity_search` embeds your query and returns the closest chunks.

In [ ]:
results = vector_store.similarity_search("What was Vestas' total revenue in 2024?", k=3)

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()

## Try it yourself 🛠️

Use `similarity_search_with_score` (instead of `similarity_search`) to get a relevance score alongside each chunk. Lower scores mean a closer match.

1. Run a **relevant** query (e.g. *"How many turbines has Vestas installed worldwide?"*) and print the scores.
2. Run an **irrelevant** query (e.g. *"recipe for chocolate cake"*) and print the scores.
3. Compare — do the irrelevant scores look noticeably worse?

In [ ]:
# Your solution here

relevant_query = "How many turbines has Vestas installed worldwide?"
irrelevant_query = "recipe for chocolate cake"

# TODO: call vector_store.similarity_search_with_score(relevant_query, k=3)
#       and print each result with its score

# TODO: do the same for irrelevant_query

# TODO: compare — what's the score range for each?